In [2]:
# ================================
# Cyberbullying Detection (2 Datasets)
# ================================

import pandas as pd
import re
import nltk
from nltk.corpus import stopwords

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

nltk.download('stopwords', quiet=True)

stop_words = set(stopwords.words('english'))

# ================================
# TEXT CLEANING FUNCTION
# ================================
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-z\s]', '', text)
    words = text.split()
    words = [w for w in words if w not in stop_words]
    return " ".join(words)

# ================================
# FUNCTION TO TRAIN MODEL
# ================================
def train_model(df, text_col, target_col, dataset_name):
    
    print(f"\n===== {dataset_name} =====")
    
    df[text_col] = df[text_col].fillna('')
    df['clean_text'] = df[text_col].apply(clean_text)
    
    X = df['clean_text']
    y = df[target_col]
    
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    vectorizer = TfidfVectorizer(max_features=5000)
    X_train_vec = vectorizer.fit_transform(X_train)
    X_test_vec = vectorizer.transform(X_test)
    
    # Naïve Bayes
    nb = MultinomialNB()
    nb.fit(X_train_vec, y_train)
    nb_pred = nb.predict(X_test_vec)
    
    print("\nNaïve Bayes:")
    print("Accuracy:", accuracy_score(y_test, nb_pred))
    print(classification_report(y_test, nb_pred))
    
    # Logistic Regression
    lr = LogisticRegression(max_iter=1000)
    lr.fit(X_train_vec, y_train)
    lr_pred = lr.predict(X_test_vec)
    
    print("\nLogistic Regression:")
    print("Accuracy:", accuracy_score(y_test, lr_pred))
    print(classification_report(y_test, lr_pred))
    
    return vectorizer, nb, lr

# ================================
# DATASET 1: TOXIC COMMENTS
# ================================
df1 = pd.read_csv(r"C:\Users\User\Desktop\archive\train.csv")  # change path if needed

# create binary label
df1['label'] = df1[['toxic','severe_toxic','obscene','threat','insult','identity_hate']].max(axis=1)

vec1, nb1, lr1 = train_model(df1, 'comment_text', 'label', "Toxic Comment Dataset")

# ================================
# DATASET 2: TWITTER HATE SPEECH
# ================================
df2 = pd.read_csv(r"C:\Users\User\Desktop\hate_dataset\labeled_data.csv")   # change path

# Convert to binary (0 = hate/offensive, 1 = normal)
df2['label'] = df2['class'].apply(lambda x: 0 if x in [0,1] else 1)

vec2, nb2, lr2 = train_model(df2, 'tweet', 'label', "Twitter Hate Speech Dataset")

# ================================
# PREDICTION FUNCTION
# ================================
def predict(text):
    print("\nInput:", text)
    
    t = clean_text(text)
    
    v1 = vec1.transform([t])
    v2 = vec2.transform([t])
    
    print("\n[Toxic Dataset Model]")
    print("NB:", "Toxic" if nb1.predict(v1)[0]==1 else "Non-Toxic")
    print("LR:", "Toxic" if lr1.predict(v1)[0]==1 else "Non-Toxic")
    
    print("\n[Twitter Dataset Model]")
    print("NB:", "Toxic" if nb2.predict(v2)[0]==0 else "Non-Toxic")
    print("LR:", "Toxic" if lr2.predict(v2)[0]==0 else "Non-Toxic")

# ================================
# TEST
# ================================
predict("You are stupid and useless")
predict("Have a great day!")


===== Toxic Comment Dataset =====

Naïve Bayes:
Accuracy: 0.948049506501645
              precision    recall  f1-score   support

           0       0.95      1.00      0.97     28671
           1       0.94      0.52      0.67      3244

    accuracy                           0.95     31915
   macro avg       0.94      0.76      0.82     31915
weighted avg       0.95      0.95      0.94     31915


Logistic Regression:
Accuracy: 0.9561334795550681
              precision    recall  f1-score   support

           0       0.96      0.99      0.98     28671
           1       0.92      0.62      0.74      3244

    accuracy                           0.96     31915
   macro avg       0.94      0.81      0.86     31915
weighted avg       0.95      0.96      0.95     31915


===== Twitter Hate Speech Dataset =====

Naïve Bayes:
Accuracy: 0.8876336493847085
              precision    recall  f1-score   support

           0       0.89      0.99      0.94      4122
           1       0.91  